# Prediction on your own molecules

This notebook assume that you have docked complexes of your molecules docked with METLL3. 

# Load Libraries

In [1]:
import numpy as np
import sys, os
sys.path.insert(0, os.path.join("/home/juni/working/mettl3/notebooks/attention_score/AttentionScore_materials_importable/AttentionScore/", "src"))

In [2]:
from attentionscore.features.plec import plec_from_dir

df_plec = plec_from_dir(
    docked_dir="/home/juni/working/mettl3/notebooks/attention_score/AttentionScore/example/docked_complexes/",       # folder with *.sdf
    protein_path="/home/juni/working/mettl3/notebooks/attention_score/AttentionScore/example/receptor.pdb",   # PDB/MOL2 etc.
    n_jobs=20,
    size=4092,
    depth_protein=4,
    depth_ligand=2,
    distance_cutoff=4.5,
    sparse=False,
    sort_numeric=True,                     # mimics your numeric filename sort
    output="numpy",                        # or "base64"/"bytes"
)
df_plec.head()

*** Open Babel Error  in openLib
  /home/juni/anaconda3/envs/Jupyter_Dock/lib/openbabel/3.1.0/png2format.so did not load properly.
 Error: /home/juni/anaconda3/envs/Jupyter_Dock/lib/openbabel/3.1.0/../.././libfontconfig.so.1: undefined symbol: FT_Done_MM_Var


,id,plec_len,plec_np
0,mol_0_docked,4092,"[1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ..."
1,mol_1_docked,4092,"[1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, ..."
2,mol_2_docked,4092,"[0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, ..."
3,mol_3_docked,4092,"[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, ..."
4,mol_4_docked,4092,"[0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, ..."


In [3]:
plec_features = np.stack(df_plec["plec_np"].values)
plec_features

array([[1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 1, 0]], dtype=uint8)

In [4]:
# If placed at src/attentionscore/predict/sdf_utils.py
from attentionscore.predict.sdf_utils import sdf_dir_to_smiles

df = sdf_dir_to_smiles(
    docked_dir="/home/juni/working/mettl3/notebooks/attention_score/AttentionScore/example/docked_complexes/",
    pattern="*.sdf",
    output_csv="/home/juni/working/mettl3/notebooks/attention_score/AttentionScore/example/docked_complexes/smiles.csv",
    output_smi="/home/juni/working/mettl3/notebooks/attention_score/AttentionScore/example/docked_complexes/smiles.smi",   # optional .smi (SMILES name)
)

[15:33:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:33:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:33:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:33:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:33:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:33:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


In [5]:
df

,file,mol_index,name,smiles
0,/home/juni/working/mettl3/notebooks/attention_...,0,mol_0,[H]O[C@@H]1[C@@H](CCc2ccc3ccc(N([H])Cc4ccccc4)...
1,/home/juni/working/mettl3/notebooks/attention_...,0,mol_1,[H]N(Cc1ccc2nc(CN([H])C(=O)c3cnc4ccccc4n3)cn2c...
2,/home/juni/working/mettl3/notebooks/attention_...,0,mol_2,[H]N(Cc1ccccc1)c1cc(N2CCC3(CC2)CN(c2ccc(CN4CCC...
3,/home/juni/working/mettl3/notebooks/attention_...,0,mol_3,[H]N(c1cc(N2CCC3(CC2)CN(c2ccc(CN4CCC(C)(C)CC4)...
4,/home/juni/working/mettl3/notebooks/attention_...,0,mol_4,[H]N(Cc1cc2ccc(Cn3cc(-c4cc(=O)n5ccccc5n4)nn3)c...
5,/home/juni/working/mettl3/notebooks/attention_...,0,mol_5,[H]N([H])c1ncnc2c1c(-c1ccn(C)n1)cn2-c1cncc(CN(...


In [6]:
from attentionscore.features.fingerprints import calculate_avalon_array

smiles = df["smiles"].astype(str).tolist()
avalon_features = np.vstack([calculate_avalon_array(smi, nBits=512) for smi in smiles])  # (N, 512)

In [7]:
avalon_features

array([[1, 0, 1, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 1, 0],
       [0, 1, 0, ..., 1, 1, 1],
       [0, 1, 0, ..., 0, 1, 0]])

In [8]:
import numpy as np, torch
from attentionscore.predict.infer import make_pred_loader, predict_proba_arrays, predict_classes_arrays,build_and_save_results
from attentionscore.nn.models import Model  # your trained architecture

In [ ]:
# Your features
XA = plec_features        # shape (N, 4092)   <-- PLEC
XB = avalon_features      # shape (N, 512)    <-- Avalon

# Load your trained model (CPU example)
device = torch.device("cpu")
model = Model(input_dim_A=4092, input_dim_B=512, n_heads=1, n_layers=0).to(device)
model.load_state_dict(torch.load("/home/juni/working/mettl3/notebooks/attention_score/models/model_FullModel.pth", map_location=device))
model.eval()

# Option 1: one-liner from arrays
probs = predict_proba_arrays(model, XA, XB, batch_size=256, device=device)  # shape (N,)
preds = predict_classes_arrays(model, XA, XB, threshold=0.5, device=device)

# Option 2: use a DataLoader (if you prefer batching control)
loader = make_pred_loader(XA, XB, batch_size=256, shuffle=False)
probs2 = predict_proba_loader(model, loader, device=device)


In [9]:
device = torch.device("cpu")  # or "cuda" if you want

ckpt_path = "/home/juni/working/mettl3/notebooks/attention_score/models/model_FullModel.pth"

# 1) Recreate the model with the SAME architecture as training
model = Model(
    input_dim_A=4092,     # PLEC size
    input_dim_B=512,     # ECFP4 size (change to 512 if Avalon)
    n_heads=1,
    n_layers=1,
    event_num= 1# whatever you used
).to(device)

# 2) Load checkpoint
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"], strict=True)
model.eval()

# 3) Predict (example with your arrays)
# probs = predict_proba_arrays(model, XA_plec_np, XB_ecfp4_np, batch_size=256, device=device)


/tmp/ipykernel_214852/3446871469.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)


Model(
  (drugEncoderA): feature_encoder(
    (layers): ModuleList(
      (0): EncoderLayer(
        (attn): MultiHeadAttention(
          (W_Q): Linear(in_features=4092, out_features=4092, bias=False)
          (W_K): Linear(in_features=4092, out_features=4092, bias=False)
          (W_V): Linear(in_features=4092, out_features=4092, bias=False)
          (fc): Linear(in_features=4092, out_features=4092, bias=False)
        )
        (AN1): LayerNorm((4092,), eps=1e-05, elementwise_affine=True)
        (l1): Linear(in_features=4092, out_features=4092, bias=True)
        (AN2): LayerNorm((4092,), eps=1e-05, elementwise_affine=True)
      )
    )
    (AN): LayerNorm((4092,), eps=1e-05, elementwise_affine=True)
    (l1): Linear(in_features=4092, out_features=2046, bias=True)
    (bn1): BatchNorm1d(2046, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (l2): Linear(in_features=2046, out_features=1023, bias=True)
    (l3): Linear(in_features=1023, out_features=2046, bias=

In [10]:
from torch import nn
# Option 1: one-liner from arrays
probs = predict_proba_arrays(model, plec_features, avalon_features, batch_size=128, device=device)  # shape (N,)
preds = predict_classes_arrays(model, plec_features, avalon_features, threshold=0.5, device=device)

In [11]:
preds

array([1, 1, 1, 1, 1, 1])

In [12]:
probs

array([0.99221134, 0.8910613 , 0.999355  , 0.60355055, 0.9589553 ,
       0.9752364 ], dtype=float32)

In [13]:
from attentionscore.predict.sdf_utils import sdf_dir_to_smiles

df_smi = sdf_dir_to_smiles(
    docked_dir="/home/juni/working/mettl3/notebooks/attention_score/AttentionScore/example/docked_complexes/",
    pattern="*_docked.sdf",
    output_csv=None,               # you can also write this to disk if you want
)

# Align order with whatever you used to build plec/avalon for prediction
names  = df_smi["name"].tolist()
smiles = df_smi["smiles"].tolist()

df_results = build_and_save_results(
    names=names,
    smiles=smiles,
    probs=probs,
    preds=preds,                   # or None to recompute via threshold
    out_csv="/home/juni/working/mettl3/notebooks/attention_score/AttentionScore/example/predictions.csv",
    )


[15:34:03] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:34:03] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:34:03] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:34:03] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:34:03] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:34:03] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


In [14]:
df_results

,molecule,smiles,probability,activity,activity_label
0,mol_2,[H]N(Cc1ccccc1)c1cc(N2CCC3(CC2)CN(c2ccc(CN4CCC...,0.9994,1,Active
1,mol_0,[H]O[C@@H]1[C@@H](CCc2ccc3ccc(N([H])Cc4ccccc4)...,0.9922,1,Active
2,mol_5,[H]N([H])c1ncnc2c1c(-c1ccn(C)n1)cn2-c1cncc(CN(...,0.9752,1,Active
3,mol_4,[H]N(Cc1cc2ccc(Cn3cc(-c4cc(=O)n5ccccc5n4)nn3)c...,0.9590,1,Active
4,mol_1,[H]N(Cc1ccc2nc(CN([H])C(=O)c3cnc4ccccc4n3)cn2c...,0.8911,1,Active
5,mol_3,[H]N(c1cc(N2CCC3(CC2)CN(c2ccc(CN4CCC(C)(C)CC4)...,0.6036,1,Active
